In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

# Variable simbólica
x = sp.symbols('x')

# Producto interno estándar (Legendre)
def inner_standard(f, g):
    return sp.integrate(f*g, (x, -1, 1))

# Producto interno con peso sqrt(1-x^2) (Chebyshev)
def inner_chebyshev(f, g):
    w = sp.sqrt(1 - x**2)
    return sp.integrate(f*g*w, (x, -1, 1))

# Gram-Schmidt ortogonalización
def gram_schmidt(basis, inner_product):
    ortho = []
    for f in basis:
        g = f
        for q in ortho:
            g -= inner_product(f, q) / inner_product(q, q) * q
        ortho.append(sp.simplify(g))
    return ortho

# --- (a) Verificación de ortogonalidad ---
f1, f2, f3 = 1, x, x**2
print("⟨1, x⟩ =", inner_standard(f1, f2))
print("⟨1, x^2⟩ =", inner_standard(f1, f3))
print("⟨x, x^2⟩ =", inner_standard(f2, f3))

# --- (b) Polinomios de Legendre ---
monomios = [x**i for i in range(10)]
ortogonales_legendre = gram_schmidt(monomios, inner_standard)

print("\nPolinomios de Legendre (sin normalizar):")
for i, p in enumerate(ortogonales_legendre):
    print(f"P{i}(x) =", p)

# --- (c) Polinomios de Chebyshev ---
ortogonales_chebyshev = gram_schmidt(monomios, inner_chebyshev)

print("\nPolinomios de Chebyshev (sin normalizar):")
for i, p in enumerate(ortogonales_chebyshev):
    print(f"T{i}(x) =", p)

# --- (d) Expansiones de h(x) ---
h = sp.sin(3*x)*(1 - x**2)

def expansion(f, basis, inner_product, n_terms=10):
    coeffs = []
    series = 0
    for k, phi in enumerate(basis[:n_terms]):
        c = inner_product(f, phi) / inner_product(phi, phi)
        coeffs.append(sp.simplify(c))
        series += c*phi
    return sp.simplify(series), coeffs

# Expansión en monomios
exp_monomios, coeffs_mono = expansion(h, monomios, inner_standard, n_terms=6)

# Expansión en Legendre
exp_legendre, coeffs_leg = expansion(h, ortogonales_legendre, inner_standard, n_terms=6)

# Expansión en Chebyshev
exp_chebyshev, coeffs_cheb = expansion(h, ortogonales_chebyshev, inner_chebyshev, n_terms=6)

print("\nExpansión en monomios (6 términos):", exp_monomios)
print("Expansión en Legendre (6 términos):", exp_legendre)
print("Expansión en Chebyshev (6 términos):", exp_chebyshev)

# --- (d.IV) Comparar errores numéricos ---
xx = np.linspace(-1, 1, 400)
h_num = sp.lambdify(x, h, 'numpy')
approx_mono = sp.lambdify(x, exp_monomios, 'numpy')
approx_leg = sp.lambdify(x, exp_legendre, 'numpy')
approx_cheb = sp.lambdify(x, exp_chebyshev, 'numpy')

err_mono = np.max(np.abs(h_num(xx) - approx_mono(xx)))
err_leg = np.max(np.abs(h_num(xx) - approx_leg(xx)))
err_cheb = np.max(np.abs(h_num(xx) - approx_cheb(xx)))

print("\nError máximo con monomios:", err_mono)
print("Error máximo con Legendre:", err_leg)
print("Error máximo con Chebyshev:", err_cheb)

# Graficar comparaciones
plt.figure(figsize=(10,6))
plt.plot(xx, h_num(xx), 'k', label="h(x) exacta")
plt.plot(xx, approx_mono(xx), 'r--', label="Exp. Monomios")
plt.plot(xx, approx_leg(xx), 'b--', label="Exp. Legendre")
plt.plot(xx, approx_cheb(xx), 'g--', label="Exp. Chebyshev")
plt.legend()
plt.title("Aproximaciones de h(x)")
plt.show()



⟨1, x⟩ = 0
⟨1, x^2⟩ = 2/3
⟨x, x^2⟩ = 0

Polinomios de Legendre (sin normalizar):
P0(x) = 1
P1(x) = x
P2(x) = x**2 - 1/3
P3(x) = x*(x**2 - 3/5)
P4(x) = x**4 - 6*x**2/7 + 3/35
P5(x) = x*(63*x**4 - 70*x**2 + 15)/63
P6(x) = x**6 - 15*x**4/11 + 5*x**2/11 - 5/231
P7(x) = x*(429*x**6 - 693*x**4 + 315*x**2 - 35)/429
P8(x) = x**8 - 28*x**6/15 + 14*x**4/13 - 28*x**2/143 + 7/1287
P9(x) = x*(12155*x**8 - 25740*x**6 + 18018*x**4 - 4620*x**2 + 315)/12155

Polinomios de Chebyshev (sin normalizar):
T0(x) = 1
T1(x) = x
T2(x) = x**2 - 1/4
T3(x) = x**3 - x/2
T4(x) = x**4 - 3*x**2/4 + 1/16
T5(x) = x*(x**4 - x**2 + 3/16)
T6(x) = x**6 - 5*x**4/4 + 3*x**2/8 - 1/64
T7(x) = x*(16*x**6 - 24*x**4 + 10*x**2 - 1)/16
T8(x) = x**8 - 7*x**6/4 + 15*x**4/16 - 5*x**2/32 + 1/256
T9(x) = x*(256*x**8 - 512*x**6 + 336*x**4 - 80*x**2 + 5)/256
